# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muneeb-th/ML-Assignment-1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Muneeb-th/ML-Assignment-1/main/data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

feature_cols = ["impressions_90d", "sessions_90d", "content_age_days",
                 "days_since_last_update", "ctr", "avg_position", "word_count"]
model_data = df.dropna(subset=feature_cols + ["client_id"]).reset_index(drop=True)
X = model_data[feature_cols]
y = model_data["is_declining"]

print(f"Feature vector built: {X.shape[0]} rows, {X.shape[1]} features")
print(X.head())

Feature vector built: 22301 rows, 7 features
   impressions_90d  sessions_90d  content_age_days  days_since_last_update  \
0             3803            17               187                      20   
1            15320             9               445                      25   
2            12581            11               141                      20   
3            19140           145               263                      14   
4             3970             5               147                      20   

    ctr  avg_position  word_count  
0  0.76          10.6      3221.0  
1  0.05          20.3      2481.0  
2  0.09          36.5      3515.0  
3  0.13          44.0      2803.0  
4  0.03           8.5      3080.0  


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- impressions_90d: rolling 90-day impressions, numeric, no missing values
  after dropna, available at prediction time (past window).
- sessions_90d: rolling 90-day GA4 sessions, numeric, some rows lack this
  (GA4 not tracked for all clients), available before prediction.
- content_age_days: days since page published, numeric, always available,
  static/slow-changing.
- days_since_last_update: days since last content edit, numeric, always
  available before prediction.
- ctr: click-through rate, numeric, occasionally 0 or implausible (>1)
  due to data artifacts noted in earlier notebooks.
- avg_position: average SERP position, numeric, available in the same
  window as other features.
- word_count: page length, numeric, static, always available.

No categorical features are used in this set. No feature here is
derived from the label or from a future window.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking my own features: testing whether a deliberately redundant
feature (a product of two existing features, similar to my Week 4
baseline rule) causes a suspicious score jump — the signature of real
leakage.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

groups = model_data["client_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

# Honest model
model = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(X.iloc[train_idx], y.iloc[train_idx])
scores_honest = model.predict_proba(X.iloc[test_idx])[:, 1]

# Suspect feature test
model_data["suspect"] = ((model_data["days_since_last_update"] >= 180).astype(int) *
                          (model_data["impressions_90d"] >= 500).astype(int) *
                          model_data["impressions_90d"])
X_leak = model_data[feature_cols + ["suspect"]]
model_leak = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
model_leak.fit(X_leak.iloc[train_idx], y.iloc[train_idx])
scores_leak = model_leak.predict_proba(X_leak.iloc[test_idx])[:, 1]

for k in (20, 50):
    print(f"Precision@{k}: honest={precision_at_k(scores_honest, y.iloc[test_idx].values, k):.3f}  "
          f"with_suspect={precision_at_k(scores_leak, y.iloc[test_idx].values, k):.3f}")

Precision@20: honest=0.550  with_suspect=0.500
Precision@50: honest=0.560  with_suspect=0.540


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded: health_score, priority_score, action_type — any
product-computed decision flag. These aren't present in this dataset,
but I exclude them on principle: using a system's own existing decision
as a feature just teaches a model to copy that decision, not find new
signal. Also excluded: client_id, content_id — used only for joining/
grouping, never as predictive features, since they're identifiers, not
signal.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.